# MECADOI stats

Questions: 

1. How many MECAs did we receive from eJP, and what do they look like?
2. For how many of these MECAs were depositions attempted, and how successful were they?
3. How many DOIs have we created?

All this information is in the MECADOI database.
I've fetched it from `s3://mecadoi-archives/batch/batch.sqlite3` to a local folder:

In [ ]:
db_uri = "sqlite:////Users/eidens/Projects/mecadoi-archives/batch/batch.sqlite3"

In [ ]:
import pandas

## Received MECAs

All files received from eJP get an entry in the `parsed_file` table.

The value for the `doi` column is read from the preprint DOI field in the MECA.
If the file is not a MECA or it doesn't have that field the value is `None`.

The file status is represented as an integer in the database.
For the mapping between status and number see `mecadoi.db.ParsedFile`.

In [ ]:
query_received_files = """
SELECT
  doi,
  SUBSTR(SUBSTR(path, 65), 0, 14) AS manuscript,
  received_at,
  status
FROM parsed_file
ORDER BY received_at
"""
received_files = pandas.read_sql_query(
    query_received_files,
    db_uri,
    parse_dates=["received_at"],
    dtype={"status": "category"},
)
received_files["status"] = received_files["status"].replace(
    [1,       10,        20,                21,           22],
    ["Valid", "Invalid", "No Preprint DOI", "No Reviews", "Duplicate"]
)
received_files["received_at_year_month"] = received_files["received_at"].dt.to_period("M")
received_files

As of 2022-11-07, all received files were MECA archives:

- Around 1/8 did not have a preprint DOI.
- Another 1/8 had the preprint DOI of a MECA archive that had already been received.
- For the rest a deposition attempt could be made.

In [ ]:
color_by_status = {
    "Valid": "green",
    "No Preprint DOI": "blue",
    "Duplicate": "orange",
    "Invalid": "red",
}
status_counts = received_files.value_counts("status")
status_counts, status_counts.plot.pie(
    colors=[color_by_status.get(x, 'grey') for x in status_counts.index],
    figsize=(11, 6),
    label="Status of received MECAs",
)

How many MECA archives are we receiving per month?

In [ ]:
counts_by_month_and_status = pandas.DataFrame({
    state: received_files[received_files["status"] == state].value_counts("received_at_year_month")
    for state in received_files["status"].unique()
})
(
    {"median # of MECAs received per month": int(received_files.value_counts("received_at_year_month").median())},
    counts_by_month_and_status.plot.bar(
        color=color_by_status,
        figsize=(15, 7),
        grid=True,
        stacked=True,
        xlabel="Month",
        ylabel="# of received MECAs",
    ),
)

## Deposition Attempts

Each attempt also gets an entry in the database, in the `deposition_attempt` table.

Only for `Valid` (see above) MECA archives is a deposition attempt made.

The attempt status (i.e. result) is also represented as an integer in the database.
For the mapping between status and number see `mecadoi.db.DepositionAttempt`.

There are multiple failure states:

- Generating the necesssary deposition file could fail. This usually points to a programming error.
- Before creating DOIs for reviews & replies we verify that what's in the MECA matches what EEB has:
    - There could already be DOIs assigned to a review or reply -> `DOIs Already Present`.
    - Or the amount of reviews and replies could differ -> `Deposition Verification Failed`.
- Finally, sending the deposition file to Crossref could fail, or they could reject it -> `Deposition Failed`.

In [ ]:
query_attempted_depositions = """
SELECT
  p.doi AS doi,
  SUBSTR(SUBSTR(p.path, 65), 0, 14) AS manuscript,
  p.received_at AS received_at,
  p.status AS file_status,
  d.id AS id_deposition_attempt,
  d.attempted_at AS attempted_at,
  d.status AS deposition_status
FROM parsed_file AS p
JOIN deposition_attempt AS d
  ON d.id_parsed_file = p.id
ORDER BY id_deposition_attempt
"""
attempted_depositions = pandas.read_sql_query(
    query_attempted_depositions,
    db_uri,
    parse_dates=[
        "received_at",
        "attempted_at",
    ],
    dtype={
        "file_status": "category",
        "deposition_status": "category",
    },
)
attempted_depositions["file_status"] = attempted_depositions["file_status"].replace(
    [1,       10,        20,      21,          22],
    ["Valid", "Invalid", "No DOI", "No Reviews", "Duplicate"]
)
attempted_depositions["deposition_status"] = attempted_depositions["deposition_status"].replace(
    [1,                      2,                      10,                  20,                               21],
    ["Deposition Succeeded", "DOIs Already Present", "Deposition Failed", "Deposition Verification Failed", "Deposition Generation Failed"]
)
attempted_depositions["attempted_at_year_month"] = attempted_depositions["attempted_at"].dt.to_period("M")
attempted_depositions

Here's a look at all deposition attempts.

It includes a transition period where we had already created "legacy" DOIs for all Review Commons reviews and replies on EEB.
Then, we ran MECADOI for all MECAs received up to that point, and these overlapped with the "legacy" DOIs: some MECAs already had DOIs for their reviews and replies from the "legacy" depositions.
That is the source of all or most of the attempts with the `DOIs Already Present` status.

Also, for all MECAs where the latest deposition attempt status is `Deposition Verification Failed` (i.e. those where the MECA contains a different number of reviews and replies than EEB had) the deposition is retried periodically.
This is supposed to catch cases where reviews and replies are going up on EEB with some kind of delay, which does happen occasionally.
However, many reviews and replies are simply not made public at all and thus produce a large amount of attempts with this status.

In [ ]:
color_by_deposition_status = {
    "Deposition Succeeded": "green",
    "Deposition Verification Failed": "blue",
    "DOIs Already Present": "orange",
    "Deposition Failed": "red",
}
deposition_status_counts = attempted_depositions.value_counts("deposition_status")
deposition_status_counts, deposition_status_counts.plot.pie(
    colors=[color_by_deposition_status.get(x, 'grey') for x in deposition_status_counts.index],
    figsize=(11, 6),
    label="Status of deposition attempts",
)

The transition period to automated MECADOI depositions was in October 2022:

In [ ]:
def plot_attempts_by_month_and_status(df):
    counts_by_month_and_status = pandas.DataFrame({
        state: df[df["deposition_status"] == state].value_counts("attempted_at_year_month")
        for state in df["deposition_status"].unique()
    })
    return counts_by_month_and_status, counts_by_month_and_status.plot.bar(
        color=color_by_deposition_status,
        figsize=(15, 7),
        grid=True,
        stacked=True,
        xlabel="Month",
        ylabel="# of depositition attempts",
    )
_ = plot_attempts_by_month_and_status(attempted_depositions)

Let's look only at deposition attempts after this date. All of these were done automatically, without user intervention.
Also filter out any "Verification Failed" attempts because these are only related to our current deposition process.

In [ ]:
cutoff_automated_depositions_year_month = "2022-09"
automated_depositions = attempted_depositions[attempted_depositions["attempted_at_year_month"] > cutoff_automated_depositions_year_month]
automated_depositions = automated_depositions[automated_depositions["deposition_status"] != "Deposition Verification Failed"]
automated_depositions

In [ ]:
automated_deposition_status_counts = automated_depositions.value_counts("deposition_status")
automated_deposition_status_counts, automated_deposition_status_counts.plot.pie(
    colors=[color_by_deposition_status.get(x, 'grey') for x in deposition_status_counts.index],
    figsize=(11, 6),
    label="Status of deposition attempts",
)

From how many MECAs per month are we depositing DOIs for reviews and replies?

In [ ]:
attempts_by_month_and_status, _ = plot_attempts_by_month_and_status(automated_depositions)
{'median # of successful deposition attempts per month': attempts_by_month_and_status["Deposition Succeeded"].median()}